In [ ]:
library(dagitty)
library(broom)
library(car)
library(ResourceSelection)
library(tidyverse)
library(emmeans)
library(mice)
library(glmnet)
library(stabs)
library(logistf)
library(ggplot2)
library(dplyr)
library(survey)
library(tmle)
library(mlr3)
library(mlr3learners)
library(ranger)
library(svglite)
library(ggcorrplot)
library(devtools)

In [ ]:
ehr <- read_csv("cohort_names.csv")

In [ ]:
cols_pe <- grep("(PE|E)$", names(ehr), value = TRUE)
cols_sdoh <- cols_pe[!cols_pe %in% c("AGE", "RACE")]
sdoh_df <- ehr[, c("ID", 'Label', cols_sdoh)]

In [ ]:
sdoh_var <- read_csv("final_sodo_var.csv") 

In [ ]:
sdoh_subdf <- sdoh_df[, c("ID", "Label", intersect(names(sdoh_df), sdoh_var$Variable))]
sdoh_subdf <- sdoh_subdf[, !grepl("^before", names(sdoh_subdf))]

In [ ]:
sdoh_2subdf <- list()
for (cat in unique(sdoh_var$category)) {
  vars_in_cat <- sdoh_var$Variable[sdoh_var$category == cat]
  vars_in_cat <- intersect(names(sdoh_subdf), vars_in_cat)  # only keep existing cols
  sdoh_2subdf[[cat]] <- sdoh_subdf[, c("ID", "Label",vars_in_cat), drop = FALSE]
}

In [ ]:
neigh_df <- sdoh_2subdf$'Neighborhood and Built Environment'
social_df <- sdoh_2subdf$'Social and Community Context'
ecomy_df <- sdoh_2subdf$'Economic Stability'
educa_df <- sdoh_2subdf$'Education Access and Quality'
insur_df <- sdoh_2subdf$'Healthcare Access and Quality'

In [ ]:
library(dplyr)

# ---------- helpers ----------
.clean_features <- function(df, id="ID", y="Label", max_na_prop=0.2) {
  feats <- setdiff(names(df), c(id, y))
  keep <- vapply(feats, function(v) {
    x <- df[[v]]
    is.numeric(x) &&
      mean(!is.finite(x)) <= max_na_prop &&
      (sd(x, na.rm=TRUE) > 0)
  }, logical(1))
  df[, c(id, y, feats[keep]), drop = FALSE]
}

.impute_suffix_aware <- function(X, pe_method="median", e_method="mean") {
  impute_vec <- function(v, how) {
    m <- if (how=="median") stats::median(v, na.rm=TRUE) else base::mean(v, na.rm=TRUE)
    if (!is.finite(m)) m <- 0
    v[!is.finite(v)] <- m
    v
  }
  for (j in seq_len(ncol(X))) {
    nm <- colnames(X)[j]
    if (grepl("PE$", nm))      X[,j] <- impute_vec(X[,j], pe_method)
    else if (grepl("E$", nm))  X[,j] <- impute_vec(X[,j], e_method)
    else                       X[,j] <- impute_vec(X[,j], "median")
  }
  X
}

.zscore <- function(X) {
  mu  <- colMeans(X, na.rm=TRUE)
  sdv <- apply(X, 2, sd, na.rm=TRUE); sdv[!is.finite(sdv) | sdv==0] <- 1
  X <- sweep(sweep(X, 2, mu, "-"), 2, sdv, "/")
  X[!is.finite(X)] <- 0
  X
}

# ---------- singel domain ----------
reduce_domain_df <- function(domain_df, label="Label",
                             max_vars=5, corr_thresh=0.8,
                             max_na_prop=0.2, do_zscore=TRUE) {
  df <- .clean_features(domain_df, id="ID", y=label, max_na_prop=max_na_prop)
  yv <- df[[label]]
  X <- df %>% dplyr::select(-all_of(c("ID", label)))
  if (ncol(X) == 0) {
    warning("没有可用变量")
    return(character(0))
  }
  
  X <- .impute_suffix_aware(as.matrix(X))
  
  if (do_zscore) X <- .zscore(X)
  
  vars <- colnames(X)
  if (length(vars) <= max_vars) {
    cat("variable no. ≤", max_vars, "，no need to reduce\n")
    return(vars)
  }
  
  # Step 1
  cor_mat <- cor(X, use="pairwise.complete.obs")
  cor_mat[!is.finite(cor_mat)] <- 0
  
  D <- 1 - abs(cor_mat)
  hc <- hclust(as.dist(D), method="average")
  groups <- cutree(hc, h=1 - corr_thresh)
  
  kept <- c()
  for (g in unique(groups)) {
    members <- names(groups)[groups==g]
    if (length(members)==1) {
      kept <- c(kept, members)
    } else {
      cors <- sapply(members, function(v) abs(cor(domain_df[[v]], yv, use="pairwise.complete.obs")))
      best <- members[which.max(cors)]
      kept <- c(kept, best)
    }
  }
  
  # Step 2
  if (length(kept) > max_vars) {
    cors <- sapply(kept, function(v) abs(cor(domain_df[[v]], yv, use="pairwise.complete.obs")))
    kept <- kept[order(-cors)][1:max_vars]
  }
  
  cat("Original vars =", length(vars), " → reduced vars =", length(kept), "\n")
  print(kept)
  
  return(kept)
}

In [ ]:
##seems to be good
insur_vars   <- reduce_domain_df(insur_df,   label="Label", max_vars=5, corr_thresh=0.75)
economy_vars <- reduce_domain_df(ecomy_df, label="Label", max_vars=5, corr_thresh=0.75)
edu_vars     <- reduce_domain_df(educa_df,     label="Label", max_vars=5, corr_thresh=0.75)
social_vars  <- reduce_domain_df(social_df,  label="Label", max_vars=5, corr_thresh=0.75)
neigh_vars   <- reduce_domain_df(neigh_df,   label="Label", max_vars=5, corr_thresh=0.75)

final_vars <- unique(c(insur_vars, economy_vars, edu_vars, social_vars, neigh_vars))
cat("variable No.", length(final_vars), "\n")

In [ ]:
####now for selected each var, call the several causal inference models####

In [ ]:
# ======================================================
# Load libraries
# ======================================================
suppressPackageStartupMessages({
  library(tidyverse)
  library(forcats)
  library(broom)
  library(tmle)
})

# ----------------- helpers -----------------
drop_high_missing <- function(df, threshold = 0.2, always_keep = character()) {
  miss_rate <- sapply(df, function(x) mean(is.na(x)))
  keep <- unique(c(names(miss_rate[miss_rate <= threshold]), always_keep))
  df[, intersect(keep, names(df)), drop = FALSE]
}

clean_race <- function(x) {
  x <- fct_na_value_to_level(as.factor(x), level = "Unknown")
  recode_factor(
    x,
    "Black or African American" = "Black",
    "White"                     = "White",
    "Asian"                     = "Asian",
    "Multiple race"             = "Multiple",
    "Native Hawaiian or Other Pacific Islander" = "NH/PI",
    "American Indian or Alaska Native"          = "AI/AN",
    "Other"                     = "Other",
    "Refuse to answer"          = "Unknown",
    "No information"            = "Unknown",
    "Unknown"                   = "Unknown"
  )
}

impute_numeric_suffix <- function(x, nm) {
  x[is.infinite(x)] <- NA_real_
  fill <- if (grepl("PE$", nm)) median(x, na.rm = TRUE) else
          if (grepl("E$",  nm)) mean(x,   na.rm = TRUE) else
                                   median(x, na.rm = TRUE)
  if (!is.finite(fill)) fill <- 0
  x[is.na(x)] <- fill
  x
}

impute_step2_better <- function(df) {
  num_cols <- names(df)[vapply(df, is.numeric, logical(1))]
  for (nm in num_cols) df[[nm]] <- impute_numeric_suffix(df[[nm]], nm)
  chr_cols <- names(df)[vapply(df, is.character, logical(1))]
  for (nm in chr_cols) df[[nm]][is.na(df[[nm]])] <- "Missing"
  fac_cols <- names(df)[vapply(df, is.factor, logical(1))]
  for (nm in fac_cols) df[[nm]] <- fct_na_value_to_level(df[[nm]], level = "Missing")
  df
}

sig_stars <- function(p) {
  if (is.na(p)) "" else if (p < 0.001) "***" else if (p < 0.01) "**"
  else if (p < 0.05) "*" else if (p < 0.1) "." else ""
}

expand_adjusters <- function(adj_set, sdoh_domains, df, adjusters_as = c("composite","indicators")) {
  adjusters_as <- match.arg(adjusters_as)
  if (length(adj_set) == 0) return(character(0))
  out <- c()
  for (a in adj_set) {
    if (adjusters_as == "indicators" && (a %in% names(sdoh_domains))) {
      out <- c(out, intersect(sdoh_domains[[a]], names(df)))
    } else {
      out <- c(out, a)
    }
  }
  out[out %in% names(df)]
}

# ======================================================
# Main Function
# ======================================================
run_causal_inference <- function(data,
                                 sdoh_domains,
                                 adj_map,
                                 model_type = c("glm"),
                                 exposure_as   = c("indicator","composite"),
                                 adjusters_as  = c("composite","indicators"),
                                 composite_method = c("mean","pca"),
                                 treat_quantile = 0.5,
                                 weight_trunc = c(0.01,0.99)) {

  model_type      <- match.arg(model_type)
  exposure_as     <- match.arg(exposure_as)
  adjusters_as    <- match.arg(adjusters_as)
  composite_method<- match.arg(composite_method)

  always_keep <- unique(c(
    "Label","AGE","SEX","RACE","Age","Sex","Genetic_testing",
    unlist(sdoh_domains),
    names(sdoh_domains),
    unlist(adj_map)
  ))

  df0 <- data %>%
    rename(Genetic_testing = Label, Age = AGE, Sex = SEX, RACE = RACE)
  df0 <- drop_high_missing(df0, threshold = 0.2, always_keep = always_keep)
  df0$Age <- suppressWarnings(as.numeric(df0$Age))
  df0$Sex <- as.factor(df0$Sex)
  df0$RACE <- clean_race(df0$RACE)
  df0$Genetic_testing <- ifelse(as.character(df0$Genetic_testing) %in% c("1","Yes","TRUE",1,TRUE), 1L, 0L)

  for (v in intersect(unlist(sdoh_domains), names(df0))) {
    if (!is.numeric(df0[[v]])) df0[[v]] <- suppressWarnings(as.numeric(df0[[v]]))
  }

  df0 <- impute_step2_better(df0)
  df0 <- drop_na(df0, Genetic_testing, Age, Sex, RACE)

  # build composites if needed
  if (exposure_as == "composite" || adjusters_as == "composite") {
    for (domain_name in names(sdoh_domains)) {
      vars <- intersect(sdoh_domains[[domain_name]], names(df0))
      if (length(vars) == 0) {
        df0[[domain_name]] <- NA_real_
      } else if (composite_method == "mean") {
        df0[[domain_name]] <- rowMeans(as.data.frame(df0[vars]), na.rm = TRUE)
      } else {
        valid_rows <- complete.cases(df0[vars])
        if (sum(valid_rows) > 1) {
          pc <- prcomp(df0[vars][valid_rows, ], scale. = TRUE)
          scores <- rep(NA_real_, nrow(df0))
          scores[valid_rows] <- pc$x[,1]
          df0[[domain_name]] <- scores
        } else {
          df0[[domain_name]] <- rowMeans(as.data.frame(df0[vars]), na.rm = TRUE)
        }
      }
    }
  }

  results <- list()

  for (domain in names(sdoh_domains)) {
    domain_vars <- intersect(sdoh_domains[[domain]], names(df0))
    exposures <- if (exposure_as == "indicator") domain_vars else domain
    base_adj <- adj_map[[domain]]; if (is.null(base_adj)) base_adj <- c("Age","Sex","RACE")
    adj_set_expanded <- expand_adjusters(base_adj, sdoh_domains, df0, adjusters_as)
    adj_set_expanded <- unique(c(intersect(c("Age","Sex","RACE"), names(df0)), adj_set_expanded))

    for (exp_var in exposures) {
      if (!(exp_var %in% names(df0))) next
      if (sd(df0[[exp_var]], na.rm = TRUE) == 0 || all(is.na(df0[[exp_var]]))) next
      zname <- paste0(exp_var, "_z")
      df0[[zname]] <- as.numeric(scale(df0[[exp_var]]))

      if (model_type == "glm") {
        fml <- as.formula(paste("Genetic_testing ~", paste(c(zname, adj_set_expanded), collapse = " + ")))
        fit <- glm(fml, data = df0, family = binomial())
        sm  <- summary(fit)$coefficients
        if (!(zname %in% rownames(sm))) next
        est <- sm[zname, "Estimate"]; se <- sm[zname, "Std. Error"]; p <- sm[zname, "Pr(>|z|)"]
        ci <- est + c(-1, 1) * 1.96 * se
        results[[length(results)+1]] <- tibble(
          Domain = domain, Exposure = exp_var, Model = "GLM",
          Effect = exp(est), CI_lower = exp(ci[1]), CI_upper = exp(ci[2]),
          p_value = p, Signif = sig_stars(p)
        )
      } 
    }
  }

  if (!length(results)) return(tibble())
  final <- bind_rows(results) %>% arrange(Domain, Exposure, Model)

  # --- Short domain labels (mask-proof) ---
  short_map <- c(
    "Economic_Stability" = "Econ",
    "Education_Access_and_Quality" = "Educ",
    "Healthcare_Access_and_Quality" = "Health",
    "Social_and_Community_Context" = "Social",
    "Neighborhood_and_Built_Environment" = "Neigh"
  )
  dom_chr <- as.character(final$Domain)
  sel <- dom_chr %in% names(short_map)
  dom_chr[sel] <- unname(short_map[dom_chr[sel]])
  final$Domain <- dom_chr

  # Rename Effect column
  names(final)[names(final) == "Effect"] <-
    ifelse(model_type %in% c("glm","ipw"), "OR", "ATE")

  final

}
    
# ======================================================
# 1. Define your SDoH domains
# ======================================================
sdoh_domains <- list(
  Economic_Stability = c("DP03_0036E", "DP04_0115E", "DP03_0045E", "DP03_0090E", "DP04_0098E"),
  Education_Access_and_Quality = c("DP02_0064PE", "DP02_0058PE", "DP02_0057PE", "DP02_0065PE", "DP02_0060PE"),
  Healthcare_Access_and_Quality = c("DP03_0117PE", "DP03_0106PE", "DP03_0107PE", "DP03_0112PE", "DP03_0118PE"),
  Social_and_Community_Context = c("DP02_0091PE", "DP02_0113PE", "DP02_0045PE", "DP02_0003PE", "DP02_0012PE"),
  Neighborhood_and_Built_Environment = c("DP04_0032PE", "DP02_0085PE", "DP04_0044PE", "DP03_0024PE", "DP04_0009PE")
)
   
# ======================================================
# 2. Define your DAG-based adjustment map
# ======================================================
adj_map <- list(
  Economic_Stability = c("Age", "Sex", "RACE", "Education_Access_and_Quality"),
  Education_Access_and_Quality = c("Age", "Sex", "RACE"),
  Healthcare_Access_and_Quality = c("Age", "Sex", "RACE", "Economic_Stability", "Education_Access_and_Quality"),
  Social_and_Community_Context = c("Age", "Sex", "RACE"),
  Neighborhood_and_Built_Environment = c("Age", "Sex", "RACE", "Economic_Stability")
)

# ======================================================
# 3. Load your cleaned EHR + SDoH dataset
# ======================================================
ehr <- read_csv("cohort_names.csv", show_col_types = FALSE)

In [ ]:
res_glm_pca <- run_causal_inference(
  data = ehr,
  sdoh_domains = sdoh_domains,
  adj_map = adj_map,
  model_type   = "glm",
  exposure_as  = "indicator",
  adjusters_as = "indicator",
  composite_method = "pca"
)

print.data.frame(res_glm_pca, row.names = FALSE)

In [ ]:
# ======================================================
# Domain-Level Average Effects (Bar Plot)
# ======================================================
suppressPackageStartupMessages({
  library(tidyverse)
  library(ggplot2)
})

# --- summarise ORs by domain ---
domain_summary <- glm_results %>%
  group_by(Domain) %>%
  summarise(
    Mean_OR  = mean(OR, na.rm = TRUE),
    Lower    = mean(CI_lower, na.rm = TRUE),
    Upper    = mean(CI_upper, na.rm = TRUE),
    n_vars   = n()
  ) %>%
  mutate(Domain = factor(Domain,
                         levels = c("Economy","Education","Healthcare","Neighborhood","Social")))

# --- domain colours ---
domain_colors <- c(
  "Economy"   = "#1f77b4",  # orange
  "Education"   = "#ff7f0e",  # blue
  "Healthcare" = "#2ca02c",  # green
  "Neighborhood"  = "#9467bd",  # purple
  "Social" = "#d62728"   # red
)

# --- plot ---
p_domain <- ggplot(domain_summary,
                   aes(x = Domain, y = Mean_OR, fill = Domain)) +
  geom_bar(stat = "identity", width = 0.6, color = "black", alpha = 0.85) +
  geom_errorbar(aes(ymin = Lower, ymax = Upper),
                width = 0.2, linewidth = 0.7) +
  geom_hline(yintercept = 1, linetype = "dashed", color = "gray40") +
  scale_fill_manual(values = domain_colors) +
  scale_y_log10() +
  labs(
       subtitle = "Mean odds ratios (log scale) ± average 95% CI",
       x = "SDoH Domain", y = "Mean Odds Ratio (log scale)") +
  theme_minimal(base_size = 14) +
  theme(legend.position = "none",
        plot.title = element_text(face = "bold"),
        axis.text = element_text(color = "black"))

# --- display and save ---
print(p_domain)

In [ ]:
# ======================================================
# Cross-Domain Correlation Heatmap
# ======================================================
suppressPackageStartupMessages({
  library(ggcorrplot)
})

# --- build a clean domain-level dataframe from ehr ---
domain_composites <- ehr %>%
  transmute(
    Economy  = rowMeans(select(., DP03_0036E, DP04_0115E, DP03_0045E,
                            DP03_0090E, DP04_0098E), na.rm = TRUE),
    Education  = rowMeans(select(., DP02_0064PE, DP02_0058PE, DP02_0057PE,
                            DP02_0065PE, DP02_0060PE), na.rm = TRUE),
    Healthcare = rowMeans(select(., DP03_0117PE, DP03_0106PE, DP03_0107PE,
                             DP03_0112PE, DP03_0118PE), na.rm = TRUE),
    Neighborhood = rowMeans(select(., DP04_0032PE, DP02_0085PE, DP04_0044PE,
                            DP03_0024PE, DP04_0009PE), na.rm = TRUE),
    Social = rowMeans(select(., DP02_0091PE, DP02_0113PE, DP02_0045PE,
                             DP02_0003PE, DP02_0012PE), na.rm = TRUE)
  ) %>%
  mutate(across(everything(), scale))

# --- correlation matrix ---
corr_mat <- cor(domain_composites, use = "pairwise.complete.obs")

p_corr <- ggcorrplot(
  corr_mat, lab = TRUE,
  colors = c("#2166ac", "white", "#b2182b"),
  outline.col = "gray60",
  ggtheme = theme_minimal(base_size = 14)
) +
  theme(plot.title = element_text(face = "bold", hjust = 0.5))

# --- display and save ---
print(p_corr)

In [ ]:
# ======================================================
# Bootstrap Robustness Distributions
# ======================================================
suppressPackageStartupMessages({
  library(tidyverse)
  library(boot)
  library(ggplot2)
})

# --- Define bootstrap function ---
bootstrap_or <- function(data, indices, domain_vars) {
  d <- data[indices, ]
  results <- map_dbl(domain_vars, function(v) {
    if (!v %in% names(d)) return(NA_real_)
    fml <- as.formula(paste("Label ~ scale(", v, ")", "+ AGE + SEX + RACE"))
    fit <- tryCatch(glm(fml, data = d, family = binomial()), error = function(e) NULL)
    if (is.null(fit)) return(NA_real_)
    exp(coef(fit)[2])  # odds ratio for the domain variable
  })
  mean(results, na.rm = TRUE)  # average OR across domain indicators
}

# --- Prepare composite-level dataset ---
domain_composites <- ehr %>%
  transmute(
    Label = Label,
    AGE = AGE,
    SEX = SEX,
    RACE = RACE,
    Economy  = rowMeans(select(., DP03_0036E, DP04_0115E, DP03_0045E,
                            DP03_0090E, DP04_0098E), na.rm = TRUE),
    Education  = rowMeans(select(., DP02_0064PE, DP02_0058PE, DP02_0057PE,
                            DP02_0065PE, DP02_0060PE), na.rm = TRUE),
    Healthcare = rowMeans(select(., DP03_0117PE, DP03_0106PE, DP03_0107PE,
                             DP03_0112PE, DP03_0118PE), na.rm = TRUE),
    Neighborhood = rowMeans(select(., DP04_0032PE, DP02_0085PE, DP04_0044PE,
                            DP03_0024PE, DP04_0009PE), na.rm = TRUE),
    Social = rowMeans(select(., DP02_0091PE, DP02_0113PE, DP02_0045PE,
                             DP02_0003PE, DP02_0012PE), na.rm = TRUE)
  )

# --- Bootstrap for each domain composite ---
set.seed(42)
bootstrap_results <- list()
for (dom in c("Economy","Education","Healthcare","Neighborhood","Social")) {
  domain_vars <- names(select(domain_composites, starts_with(dom)))
  boot_out <- boot(domain_composites, statistic = function(d,i)
    bootstrap_or(d, i, domain_vars),
    R = 1000)
  bootstrap_results[[dom]] <- tibble(Domain = dom, OR_boot = boot_out$t)
}

# --- Combine all bootstrap draws ---
boot_tbl <- bind_rows(bootstrap_results)

# --- Colors ---
domain_colors <- c(
  "Economy"   = "#1f77b4",  # orange
  "Education"   = "#ff7f0e",  # blue
  "Healthcare" = "#2ca02c",  # green
  "Neighborhood"  = "#9467bd",  # purple
  "Social" = "#d62728"   # red
)
                   
# --- Plot bootstrap distributions ---
p_boot <- ggplot(boot_tbl, aes(x = OR_boot, fill = Domain)) +
  geom_density(alpha = 0.65) +
  geom_vline(xintercept = 1, color = "gray40", linetype = "dashed") +
  scale_fill_manual(values = domain_colors) +
  scale_x_log10() +
  labs(
    subtitle = "Stability of average odds ratios across 1000 resamples",
    x = "Bootstrap Odds Ratio (log scale)",
    y = "Density"
  ) +
  theme_minimal(base_size = 14) +
  theme(
    legend.position = "top",
    legend.title = element_blank(),
    plot.title = element_text(face = "bold")
  )

print(p_boot)

In [ ]:
# =========================================================
# Stratified Causal Inference Framework
# =========================================================
suppressPackageStartupMessages({
  library(tidyverse)
  library(broom)
  library(forcats)
  library(tmle)
})

# ---------- helpers ----------
sig_stars <- function(p) {
  if (is.na(p)) "" else if (p < 0.001) "***" else if (p < 0.01) "**"
  else if (p < 0.05) "*" else if (p < 0.1) "." else ""
}

impute_numeric <- function(x) {
  x[is.infinite(x)] <- NA_real_
  fill <- suppressWarnings(median(x, na.rm = TRUE))
  if (!is.finite(fill)) fill <- 0
  x[is.na(x)] <- fill
  as.numeric(scale(x))
}

clean_race <- function(x) {
  x <- fct_na_value_to_level(as.factor(x), level = "Unknown")
  recode_factor(
    x,
    "Black or African American" = "Black",
    "White"                     = "White",
    "Asian"                     = "Asian",
    "Multiple race"             = "Multiple",
    "Native Hawaiian or Other Pacific Islander" = "NH/PI",
    "American Indian or Alaska Native"          = "AI/AN",
    "Other"                     = "Other",
    "Refuse to answer"          = "Unknown",
    "No information"            = "Unknown",
    "Unknown"                   = "Unknown"
  )
}

# ---------- main ----------
run_stratified_causal_table_flex <- function(
  data,
  sdoh_domains,
  adj_map,
  model_type = c("glm", "ipw", "aipw", "tmle"),
  stratify_by = NULL,
  exposure_as = c("composite", "indicator"),
  adjusters_as = c("indicators", "composite"),
  composite_method = c("mean", "pca")
) {

  model_type       <- match.arg(model_type)
  exposure_as      <- match.arg(exposure_as)
  adjusters_as     <- match.arg(adjusters_as)
  composite_method <- match.arg(composite_method)

  # ----- preprocess -----
  df0 <- data %>%
    rename(Genetic_testing = Label,
           Age = AGE, Sex = SEX, RACE = RACE, CancerType = Cancers) %>%
    mutate(
      Genetic_testing = ifelse(as.character(Genetic_testing) %in% c("1","Yes","TRUE",1,TRUE), 1L, 0L),
      Age = suppressWarnings(as.numeric(Age)),
      Sex = as.factor(Sex),
      RACE = clean_race(RACE)
    ) %>%
    drop_na(Genetic_testing, Age, Sex, RACE)

  # Create AgeGroup
  if (!"AgeGroup" %in% names(df0)) {
    message("Creating AgeGroup variable (<40, 40–59, 60–79, ≥80)")
    df0 <- df0 %>%
      mutate(
        AgeGroup = case_when(
          Age < 40 ~ "<40",
          Age >= 40 & Age < 60 ~ "40–59",
          Age >= 60 & Age < 80 ~ "60–79",
          Age >= 80 ~ "≥80",
          TRUE ~ NA_character_
        )
      )
  }

  factor_cols <- intersect(c("RACE","Sex","AgeGroup","CancerType"), names(df0))
  for (fc in factor_cols) df0[[fc]] <- as.factor(df0[[fc]])

  # Impute & scale all SDoH indicators
  for (vars in sdoh_domains) {
    for (v in vars) if (v %in% names(df0)) df0[[v]] <- impute_numeric(df0[[v]])
  }

  # Build composite adjusters if requested
  if (adjusters_as == "composite") {
    for (domain in names(sdoh_domains)) {
      vars <- intersect(sdoh_domains[[domain]], names(df0))
      if (!length(vars)) next
      comp_name <- paste0(domain, "_comp")
      if (composite_method == "mean") {
        df0[[comp_name]] <- rowMeans(df0[, vars, drop = FALSE], na.rm = TRUE)
      } else {
        ok <- is.finite(rowSums(as.matrix(df0[, vars, drop = FALSE])))
        if (sum(ok) >= 3) {
          pc <- tryCatch(prcomp(df0[ok, vars, drop = FALSE], scale. = TRUE), error = function(e) NULL)
          if (!is.null(pc)) {
            tmp <- rep(NA_real_, nrow(df0))
            tmp[ok] <- pc$x[, 1]
            df0[[comp_name]] <- tmp
          } else {
            df0[[comp_name]] <- rowMeans(df0[, vars, drop = FALSE], na.rm = TRUE)
          }
        } else {
          df0[[comp_name]] <- rowMeans(df0[, vars, drop = FALSE], na.rm = TRUE)
        }
      }
      df0[[comp_name]] <- impute_numeric(df0[[comp_name]])
    }
  }

  # ---------- single causal model ----------
  run_one <- function(df, domain, var, drop_adjuster = NULL) {
    adj_vars <- adj_map[[domain]] %||% c("Age","Sex","RACE")
    if (!is.null(drop_adjuster)) adj_vars <- setdiff(adj_vars, drop_adjuster)

    if (adjusters_as == "composite") {
      adj_vars <- sapply(adj_vars, function(a) {
        if (a %in% names(sdoh_domains)) paste0(a, "_comp") else a
      })
    }
    adj_vars <- intersect(adj_vars, names(df))

    if (nrow(df) < 30 || length(unique(df$Genetic_testing)) < 2)
      return(tibble(Variable = var, Effect = NA_real_, p_value = NA_real_, Signif = ""))

    # ---- GLM ----
    if (model_type == "glm") {
      fml <- as.formula(paste("Genetic_testing ~", paste(c(var, adj_vars), collapse = " + ")))
      fit <- tryCatch(glm(fml, data = df, family = binomial()), error = function(e) NULL)
      if (is.null(fit)) return(tibble(Variable = var, Effect = NA, p_value = NA, Signif = ""))
      sm <- broom::tidy(fit, conf.int = TRUE, exponentiate = TRUE)
      row_v <- sm %>% filter(term == var)
      return(tibble(Variable = var, Effect = row_v$estimate, p_value = row_v$p.value, Signif = sig_stars(row_v$p.value)))
    }

    tibble(Variable = var, Effect = NA, p_value = NA, Signif = "")
  }

  # ---------- domain runner ----------
  run_on_frame <- function(subdf, drop_adjuster = NULL) {
    out <- list()
    for (domain in names(sdoh_domains)) {
      vars <- intersect(sdoh_domains[[domain]], names(subdf))
      if (!length(vars)) next

      # exposure as composite
      if (exposure_as == "composite") {
        if (composite_method == "mean") {
          W <- rowMeans(subdf[, vars, drop = FALSE], na.rm = TRUE)
        } else {
          ok <- is.finite(rowSums(as.matrix(subdf[, vars, drop = FALSE])))
          if (sum(ok) >= 3) {
            pc <- tryCatch(prcomp(subdf[ok, vars, drop = FALSE], scale. = TRUE), error = function(e) NULL)
            if (!is.null(pc)) {
              W <- rep(NA_real_, nrow(subdf))
              W[ok] <- pc$x[, 1]
            } else {
              W <- rowMeans(subdf[, vars, drop = FALSE], na.rm = TRUE)
            }
          } else {
            W <- rowMeans(subdf[, vars, drop = FALSE], na.rm = TRUE)
          }
        }
        subdf$Composite <- impute_numeric(W)
        res <- run_one(subdf, domain, "Composite", drop_adjuster) %>%
          mutate(Category = domain, Method = paste("Composite", composite_method))
        out[[length(out) + 1]] <- res
      }

      # exposure as indicator
      if (exposure_as == "indicator") {
        for (v in vars) {
          res <- run_one(subdf, domain, v, drop_adjuster) %>%
            mutate(Category = domain, Method = "Indicator-level")
          out[[length(out) + 1]] <- res
        }
      }
    }
    bind_rows(out)
  }

  # ---------- stratified execution ----------
  if (!is.null(stratify_by)) {
    found_col <- grep(paste0("^", stratify_by, "$"), names(df0), ignore.case = TRUE, value = TRUE)
    if (length(found_col) == 1) {
      stratify_by <- found_col
      message(paste("Running stratified analysis by:", stratify_by))
      df0[[stratify_by]] <- forcats::fct_explicit_na(as.factor(df0[[stratify_by]]), na_level = "Unknown")
      split_list <- split(df0, df0[[stratify_by]], drop = TRUE)
      res_list <- lapply(names(split_list), function(level) {
        subdf <- split_list[[level]]
        if (nrow(subdf) < 30 || length(unique(subdf$Genetic_testing)) < 2) return(NULL)
        res <- run_on_frame(subdf, drop_adjuster = stratify_by)
        if (is.null(res) || nrow(res) == 0) return(NULL)
        res[[stratify_by]] <- level
        res
      })
      results <- bind_rows(Filter(Negate(is.null), res_list))
    } else {
      message("Stratify variable not found; running pooled analysis.")
      results <- run_on_frame(df0, drop_adjuster = NULL)
    }
  } else {
    message("Running pooled (non-stratified) analysis.")
    results <- run_on_frame(df0, drop_adjuster = NULL)
  }

  if (is.null(results) || !nrow(results)) return(tibble())

  # ---------- formatting ----------
  short_map <- c(
    "Economic_Stability" = "Economy",
    "Education_Access_and_Quality" = "Education",
    "Healthcare_Access_and_Quality" = "Healthcare",
    "Social_and_Community_Context" = "Social",
    "Neighborhood_and_Built_Environment" = "Neighborhood"
  )
  results$Category <- ifelse(results$Category %in% names(short_map),
                             short_map[results$Category], results$Category)
  names(results)[names(results) == "Effect"] <-
    ifelse(model_type %in% c("glm", "ipw"), "OR", "ATE")

  base_cols <- c("Category","Variable","Method",
                 ifelse(model_type %in% c("glm","ipw"),"OR","ATE"),
                 "p_value","Signif")
  if (!is.null(stratify_by) && stratify_by %in% names(results))
    base_cols <- c(stratify_by, base_cols)

  results <- results %>%
    select(any_of(base_cols)) %>%
    arrange(Category, is.na(p_value), p_value)

  print(results, n = 100, width = Inf)
  invisible(results)
}

In [ ]:
tbl_glm_age_pca <- run_stratified_causal_table_flex(
  data = ehr,
  sdoh_domains = sdoh_domains,
  adj_map = adj_map,
  model_type = "glm",
  stratify_by = "AgeGroup", # "RACE","Sex","AgeGroup","CancerType"
  exposure_as = "indicator",       # "indicator" composite
  adjusters_as = "composite",     # "composite" indicator 
  composite_method = "pca"         # mean pca
)

In [ ]:
tbl_glm_sex_pca <- run_stratified_causal_table_flex(
  data = ehr,
  sdoh_domains = sdoh_domains,
  adj_map = adj_map,
  model_type = "glm",               #"glm", "ipw", "aipw", "tmle"
  stratify_by = "Sex",             #"RACE","Sex","AgeGroup","CancerType"
  exposure_as = "indicator",       # or "indicator" composite
  adjusters_as = "composite",     # or "composite" indicator 
  composite_method = "pca"         # mean pca
)

In [ ]:
###ML results###

In [ ]:
# ==============================================================
#  Random Forest Performance Comparison by Data Type
# ==============================================================

library(ggplot2)
library(reshape2)

# -----------------------------
# Define data
# -----------------------------
df <- data.frame(
  Data = c("Full", "Demographics", "SDoH", "Lab results"),
  Accuracy = c(0.793, 0.547, 0.703, 0.770),
  F1_Score = c(0.404, 0.252, 0.448, 0.379),
  AUROC = c(0.776, 0.493, 0.701, 0.714),
  AUPRC = c(0.523, 0.228, 0.441, 0.453),
  Sensitivity = c(0.307, 0.333, 0.527, 0.307),
  Specificity = c(0.937, 0.611, 0.755, 0.907)
)

# Melt to long format
df_melt <- melt(df, id.vars = "Data",
                variable.name = "Metric",
                value.name = "Score")

# -----------------------------
# Faceted plot
# -----------------------------
p <- ggplot(df_melt, aes(x = Data, y = Score, fill = Data)) +
  geom_bar(stat = "identity", width = 0.6, color = "black", alpha = 0.9) +
  geom_text(aes(label = round(Score, 3)), vjust = -0.3, size = 3.5, fontface = "bold") +
  scale_fill_brewer(palette = "Set2") +
  facet_wrap(~ Metric, nrow = 1, scales = "free_y") +  # 1 row, 6 columns
  theme_bw(base_size = 13) +
  labs(
    # title = "Panel M | Random Forest Performance Across Data Types",
    # subtitle = "Each facet shows performance for a different metric",
    x = NULL, y = "Score (0–1 scale)", fill = "Data Type"
  ) +
  ylim(0, 1.05) +
  theme(
    plot.title = element_text(face = "bold", size = 14, hjust = 0),
    plot.subtitle = element_text(size = 12, color = "gray30"),
    axis.text.x = element_text(angle = 30, hjust = 1, size = 11),
    axis.title.y = element_text(size = 12),
    legend.position = "top",
    panel.grid.major.y = element_line(color = "gray85", linetype = "dashed"),
    strip.background = element_rect(fill = "gray90", color = NA),
    strip.text = element_text(face = "bold", size = 12)
  )

# Show plot
print(p)

In [ ]:
# ==============================================================
# Decision Tree Performance Comparison by Data Type
# ==============================================================

library(ggplot2)
library(reshape2)

# -----------------------------
# Input metrics
# -----------------------------
df_dt <- data.frame(
  Data = c("Full", "Demographics", "SDoH", "Lab results"),
  Accuracy = c(0.750, 0.506, 0.657, 0.697),
  F1_Score = c(0.442, 0.293, 0.450, 0.321),
  AUROC = c(0.639, 0.494, 0.678, 0.504),
  AUPRC = c(0.325, 0.229, 0.371, 0.278),
  Sensitivity = c(0.433, 0.447, 0.613, 0.313),
  Specificity = c(0.844, 0.524, 0.670, 0.810)
)

# Melt to long format
df_melt <- melt(df_dt, id.vars = "Data",
                variable.name = "Metric", value.name = "Score")

# -----------------------------
# Plot
# -----------------------------
p_dt <- ggplot(df_melt, aes(x = Data, y = Score, fill = Data)) +
  geom_bar(stat = "identity", width = 0.6, color = "black", alpha = 0.9) +
  geom_text(aes(label = round(Score, 3)), vjust = -0.3, size = 3.5, fontface = "bold") +
  scale_fill_brewer(palette = "Set2") +
  facet_wrap(~ Metric, nrow = 1, scales = "free_y") +
  theme_bw(base_size = 13) +
  labs(
    # title = "Panel N | Decision Tree Performance Across Data Types",
    # subtitle = "Each facet shows performance for a distinct evaluation metric",
    x = NULL, y = "Score (0–1 scale)", fill = "Data Type"
  ) +
  ylim(0, 1.05) +
  theme(
    plot.title = element_text(face = "bold", size = 14, hjust = 0),
    plot.subtitle = element_text(size = 12, color = "gray30"),
    axis.text.x = element_text(angle = 30, hjust = 1, size = 11),
    axis.title.y = element_text(size = 12),
    legend.position = "top",
    panel.grid.major.y = element_line(color = "gray85", linetype = "dashed"),
    strip.background = element_rect(fill = "gray90", color = NA),
    strip.text = element_text(face = "bold", size = 12)
  )

# Display the plot
print(p_dt)